# 16 — Train XGBoost Models (Multi-Outcome)

Trains one XGBoost classifier per instability outcome using the feature sets
selected by Notebook 15. One model per outcome — no stacking.

## Relationship to existing classifier notebook

| Component | Treatment |
|---|---|
| `temporal_split()` | Reused (train ≤2018, val 2019–2021, test ≥2022) |
| `RandomizedSearchCV` loop | Reused, one call per outcome with temporal CV |
| SHAP TreeExplainer | Reused, one call per outcome |
| MLflow logging | Reused, one `mlflow.start_run()` per outcome |
| Data loading | Reads ADLS parquet (not raw CSV) |
| Feature engineering | Done in Notebook 14 — not repeated here |
| StratifiedKFold | **Replaced** with temporal CV to prevent future-leakage |

## Class imbalance

| Outcome | Expected base rate | `scale_pos_weight` |
|---|---|---|
| `civil_war_onset` | ~0.2% | ~500 |
| `coup_attempt` | ~0.1–0.3% | ~300–1000 |
| `regime_backsliding` | ~1% | ~100 |
| `mass_unrest_onset` | ~12–20% | ~4–8 |
| `humanitarian_crisis_onset` | ~5–10% (regional) | ~10–20 |

Primary metric: **AUPRC** (average precision). AUROC also logged.

## ADLS outputs per outcome
```
models/{RUN_DATE}/{outcome}/model.json
models/{RUN_DATE}/{outcome}/shap_values.parquet
```

## Required environment variables
```
ADLS_ACCOUNT_NAME
ADLS_CONTAINER       (default: 'data')
AZUREML_MLFLOW_URI   (optional)
```

In [ ]:
import os
import json
import re
import tempfile
import warnings
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

from sklearn.model_selection import RandomizedSearchCV, BaseCrossValidator
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.calibration import CalibrationDisplay

import xgboost as xgb
import shap

from azure.identity import DefaultAzureCredential
import adlfs
import mlflow
import mlflow.xgboost

warnings.filterwarnings('ignore', category=FutureWarning)
pd.set_option('display.max_columns', 40)

## Configuration

In [ ]:
ADLS_ACCOUNT_NAME = os.environ["ADLS_ACCOUNT_NAME"]
ADLS_CONTAINER    = os.getenv("ADLS_CONTAINER", "data")
RUN_DATE          = datetime.utcnow().strftime("%Y%m%d")

# Temporal split boundaries (must match notebooks 14 and 15)
TRAIN_END_YEAR    = 2018
VAL_END_YEAR      = 2021

OUTCOMES = [
    "civil_war_onset",
    "coup_attempt",
    "regime_backsliding",
    "mass_unrest_onset",
    "humanitarian_crisis_onset",
]

# Hyperparameter search
N_ITER_SEARCH     = 40   # reduced to 20 for humanitarian (small N)
N_ITER_HUMANITARIAN = 20
RANDOM_STATE      = 42

# XGBoost base params (overridden by search)
XGB_BASE_PARAMS = {
    "objective":     "binary:logistic",
    "eval_metric":   "aucpr",
    "tree_method":   "hist",
    "random_state":  RANDOM_STATE,
    "n_jobs":        -1,
    "verbosity":     0,
}

# Hyperparameter search space
PARAM_DIST = {
    "n_estimators":      [100, 200, 300, 500],
    "max_depth":         [3, 4, 5, 6],
    "learning_rate":     [0.01, 0.05, 0.1, 0.2],
    "subsample":         [0.6, 0.7, 0.8, 1.0],
    "colsample_bytree":  [0.6, 0.7, 0.8, 1.0],
    "min_child_weight":  [1, 3, 5, 10],
    "gamma":             [0, 0.1, 0.3, 1.0],
    "reg_alpha":         [0, 0.01, 0.1, 1.0],
    "reg_lambda":        [1.0, 2.0, 5.0],
}

print(f"Run date       : {RUN_DATE}")
print(f"Temporal split : train ≤{TRAIN_END_YEAR} | val ≤{VAL_END_YEAR} | test ≥{VAL_END_YEAR+1}")